In [ ]:
# ===== Environment setup - run me FIRST =====
# Set where this notebook runs: at most ONE of the two flags True (both False = local).
#
#   Kaggle  attach the code and data datasets built by
#           `python -m backend.notebooks.shared.kaggle bundle` (Add Input). Nothing else to edit.
#   Colab   upload the WHOLE field-disagreement-prediction folder to Drive and set DRIVE_ROOT
#           if it is not MyDrive/field-disagreement-prediction.
#   Local   leave both False.
#
# The cohort is read from configs/notebooks/base.yaml, the file the training cells read,
# so staging cannot disagree with training (backend/notebooks/shared/{kaggle,colab}.py).
RUN_ON_KAGGLE = False
RUN_ON_COLAB = False
RELEASE_RUNTIME_AT_END = True      # Colab only: disconnect the GPU after the last cell
DRIVE_ROOT = '/content/drive/MyDrive/field-disagreement-prediction'   # Colab only

import os
import sys
from pathlib import Path

assert not (RUN_ON_KAGGLE and RUN_ON_COLAB), "set at most one of RUN_ON_KAGGLE / RUN_ON_COLAB"
os.environ["FDP_PLATFORM"] = "kaggle" if RUN_ON_KAGGLE else "colab" if RUN_ON_COLAB else "local"

if RUN_ON_KAGGLE:
    # Find the uploaded code by content, whatever the dataset is called.
    CODE = next((p.parent for depth in range(1, 6)
                 for p in sorted(Path("/kaggle/input").glob("*/" * depth + "pyproject.toml"))
                 if (p.parent / "backend" / "notebooks" / "shared" / "kaggle.py").exists()),
                None)
    if CODE is None:
        raise FileNotFoundError("no field-disagreement-prediction code under /kaggle/input - Add Input > "
                                "Your Datasets > fdp-code (see backend/notebooks/shared/kaggle.py)")
    sys.path.insert(0, str(CODE))
    from backend.notebooks.shared.kaggle import stage_kaggle
    ENV = stage_kaggle(CODE, config_name="notebooks/study")   # code -> scratch, cohort, runs restored, time budget
elif RUN_ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, DRIVE_ROOT)                             # the stager is imported from the Drive copy
    from backend.notebooks.shared.colab import stage
    ENV = stage(DRIVE_ROOT, config_name="notebooks/study")     # code and cohort -> local disk; re-points sys.path
else:
    ENV = None
    print("local run - nothing to stage")


# Field-level prediction of expert-panel disagreement

Several radiologists (LIDC-IDRI) or ophthalmologists (Chákṣu) read every case, and they often disagree.
A triage system that only looks at its own confidence cannot tell *the model is unsure* from *the experts
would argue about this case*. The study trains the network to predict, from the image alone, **how much the
panel will disagree on every field of its report**, and uses that prediction to keep disputed cases with
a human reader.

**What already exists, and what is ours**

| Method | Name in the paper | What it learns from the panel | Routes cases on |
|---|---|---|---|
| `A_cls` | Confidence | nothing (consensus label only) | classifier confidence |
| `B_kl` | Soft-KL | the panel's rating histogram on the diagnosis (KL loss) | histogram spread |
| `C_emd` | Soft-EMD | the same histogram (ordinal EMD loss) | histogram spread |
| `D_reg` | DUP | the panel's disagreement on the diagnosis (Raghu et al. 2019) | predicted disagreement |
| `R_rr` | Random-reader | one randomly drawn reader's label per step | classifier confidence |
| `U_edl` | Evidential | an evidential Beta over the diagnosis (Sensoy et al. 2018) | vacuity |
| `P_emd` | Field-soft | a rating histogram for **every** report field (strongest baseline) | mean spread over fields |
| **`H_fld`** | **Field-disp (ours)** | Field-soft **plus** one output per field that regresses that field's disagreement | mean of its per-field outputs |
| `H_fldp` | Field-disp shuffled | ours, trained on another case's panel (control) | mean of its outputs |

**Why ours should help.** A disagreement read off a predicted histogram is a by-product of fitting the
histogram. Field-disp is trained directly on what a deferral signal needs: which cases, and which fields,
the panel will argue about. The shuffled control shows whether the gain comes from each case's own panel.

**How it is measured.** For each report field, the Spearman correlation between the method's signal for
that field and the panel's real disagreement, averaged over fields, on held-out patients (five folds,
three training seeds). Then a simulated triage rule shows what each signal does to the cases a reader sees.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Make the repository importable from backend/notebooks (local) or the staged root (Colab, Kaggle).
ROOT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from field_disagreement import ExperimentConfig
from field_disagreement import metrics as M
from field_disagreement.triage.ladder import ABLATIONS, ALL_ARMS, DISPLAY_NAMES, ships
from backend.config import load, paths
from backend.domains import datasets as DS
from backend.notebooks.shared import (CheckpointStore, FoldRunner, cluster_map, demo,
                                      manifest_and_folds, run_scope, save_summary,
                                      screen_arms, set_seed)
from backend.notebooks.shared import study_arms as SARM
from backend.notebooks.shared import study_external as EXT
from backend.notebooks.shared import study_results as SRES

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## Configuration

Every setting is in `configs/notebooks/study.yaml`, which extends `configs/notebooks/base.yaml`.
The cohort is chosen in `base.yaml` (`extends: cohorts/lidc` or `cohorts/chaksu`); the data folders are
in `configs/paths.yaml`. This cell only reads them and derives the device and the output folder.

In [ ]:
P = paths()
cfg = load("notebooks/study")
SCOPE = run_scope(cfg)                      # folds to run; a smoke run gets its own run_dir
FOLDS = SCOPE.folds
spec = DS.bind(cfg)                         # processed folder, frozen split, contested cut
spec.require_capabilities(*ExperimentConfig.from_namespace(cfg).needed_capabilities())
cfg.out = P.runs / cfg.run_dir
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"

RUN_ARMS = screen_arms(cfg, ALL_ARMS, optional=ABLATIONS)
EXPERIMENT = ExperimentConfig.from_namespace(cfg)
SEEDS = list(cfg.seeds)
DEMO = cfg.demo

set_seed(cfg.seed)
device = torch.device(cfg.device)
Path(cfg.out).mkdir(parents=True, exist_ok=True)
print(f"cohort {spec.name} | model {cfg.model} | device {device} | folds {FOLDS} | seeds {SEEDS}")
print("arms:", ", ".join(f"{a} ({DISPLAY_NAMES[a]})" for a in RUN_ARMS))
print(f"contested cut d0 = {cfg.contested_d} | output: {cfg.out}")
if SCOPE.note:
    print(SCOPE.note)

## The data

Each processed item is one `.npz` file: the image, the readers' masks, the consensus label, and the
panel's disagreement on every report field (`d_profile`, in units of that field's spread over the cohort).
Patients, not images, are split into five folds that were fixed before any model was trained.

In [ ]:
df, patient_fold = manifest_and_folds(cfg)
demo.describe_cohort(df, cfg.processed_dir, spec.field_names)

### What the panel disagrees about

The same image can be unanimous on one field and disputed on another. The cases below run from the least
to the most disputed; the bars are the panel's disagreement on each report field.

In [ ]:
files = demo.pick_by_disagreement(cfg.processed_dir, DEMO["gallery_items"], files=df["file"])
demo.panel_gallery(cfg.processed_dir, files, spec.field_names,
                   title=f"{spec.name}: panel disagreement per report field")
plt.show()

## Checkpoints

Free GPU sessions (Colab, Kaggle) can stop at any time. Every epoch the full training state is saved,
so an interrupted fold resumes where it stopped, and a finished fold is loaded instead of retrained.
`runs:` in `configs/paths.yaml` points at storage that survives the session (Kaggle output, Google Drive,
or a local folder). The selected weights of every arm are kept (`keep_best`), so the figures at the end
can be redrawn without training.

In [ ]:
CKPT = CheckpointStore(Path(cfg.out) / "checkpoints", cfg.model, device,
                       keep_best=bool(cfg.keep_best), keep_ckpt=bool(cfg.keep_ckpt))

## Train every arm

Every arm trains the same network with the same schedule, folds and checkpoint-selection rule; only the
panel targets differ. Each seed is a full pass over all folds. After fold 0, a check confirms that every
panel-trained arm's validation signal had reached 90% of its best at the kept epoch; a shortfall is
printed and reported, not hidden.

In [ ]:
RUNNER = FoldRunner(cfg, CKPT, device, df, patient_fold, schema=SARM.STUDY_SCHEMA,
                    experiment=EXPERIMENT)
CLUSTERS = cluster_map(df)                       # patient of each item, for the bootstrap
FLOOR = int(cfg.defer_warmup * cfg.epochs)       # earliest epoch selection may keep
FC = cfg.floor_check

BY_SEED = {}
for SEED in SEEDS:
    cfg.seed = SEED
    exp = replace(EXPERIMENT, seed=SEED)
    RUNNER.experiment = exp
    print(f"\n===== seed {SEED} =====")
    by_arm = {a: [] for a in RUN_ARMS}
    for i, f in enumerate(FOLDS):                # every arm finishes a fold before the next
        for a in RUN_ARMS:
            by_arm[a].append(SARM.run_fold(RUNNER, cfg, exp, a, f, clusters=CLUSTERS,
                                          device=device))
        if i == 0 and not SCOPE.smoke:
            fc = SRES.floor_check({f: {a: by_arm[a][0] for a in RUN_ARMS}}, floor=FLOOR,
                                 share=FC["share"], max_floor=FC["max_floor"],
                                 window=exp.training.sel_window)
            SRES.print_floor_check(fc, FC["share"])
            if not fc["ok"] and FC["on_fail"] == "report":
                print(f"  -> continuing at the declared floor (epoch {FLOOR}); "
                      "these units are reported, with results also shown without them")
            elif not fc["ok"]:
                raise RuntimeError(
                    f"floor check failed: set defer_warmup to {fc['floor'] / cfg.epochs:g} "
                    f"in configs/notebooks/base.yaml and a new run_dir, then run again")
    suffix = "" if SEED == SRES.BASE_SEED else f"_s{SEED}"
    save_summary(Path(cfg.out) / f"{cfg.model}_summary_nb{suffix}.json", cfg.model, by_arm,
                 {a: M.aggregate_folds(r) for a, r in by_arm.items()})
    BY_SEED[SEED] = by_arm
cfg.seed = SEEDS[0]
by_arm = BY_SEED[SEEDS[0]]
CKPT.sweep()                                     # clear anything a Drive write left in Trash

## Results

Everything below is read back from the run folder, so it includes seeds trained in earlier sessions.

1. **One seed, every table** - primary metric and contrasts against every arm, secondary metrics,
   field by field, classification AUC, and the triage analysis.
2. **All seeds pooled** - the primary contrasts with the three seeds pooled draw by draw.
3. **Selection check** - the same networks at the epoch validation AUC alone would keep.
4. **External test** (LIDC study only) - the LIDC-trained networks, unchanged, on LNDb.
5. **Ablation** (only when this run trained it) - the claim beside its per-field
   disagreement outputs alone (`H_disp`), all seeds pooled.

A run may train only some arms (`screen_arms`). `compare_runs` in `study.yaml` then names
the study's finished run folders: the tables read them together with this run's folder, and
where both hold an arm, this run's result is used.

In [ ]:
from backend.notebooks.shared import study_summary

# This run's folder first, then the finished study folders named in `compare_runs`.
# Where both hold the same arm, this run's result is the one read.
READ = [Path(cfg.out), *(Path(P.runs) / r for r in cfg.compare_runs)]
print("folders read:", [p.name for p in READ])
SRES.result_files(READ, verbose=True)
DONE = SRES.seeds_in(READ)                    # every seed with a result for the claim
BASE = SRES.BASE_SEED if SRES.BASE_SEED in DONE or not DONE else DONE[0]
print("seeds with a result for the claim:", DONE)

REPORT = SEEDS_REPORT = SENS_REPORT = EXTERNAL_REPORT = None
if not DONE:
    print(f"no {SRES.CLAIM} result in these folders: a run that trains only some arms "
          "needs the study's run folders in compare_runs")
else:
    print(f"\n{'=' * 28} 1. seed {BASE}: every table {'=' * 28}")
    REPORT = SRES.report(READ, d0=cfg.contested_d, seed=BASE)

    if len(DONE) > 1:
        print(f"\n{'=' * 28} 2. all seeds pooled {'=' * 28}")
        SEEDS_REPORT = SRES.report_seeds(READ, DONE)

    if cfg.sel_also:
        print(f"\n{'=' * 28} 3. selection check: AUC-only epoch {'=' * 28}")
        SENS_REPORT = SRES.report(READ, d0=cfg.contested_d, seed=BASE,
                                 selection="sensitivity")

if spec.name == "lidc_report":
    print(f"\n{'=' * 28} 4. external test on LNDb {'=' * 28}")
    try:
        LNDB = EXT.find("lndb_report")
        for s in SEEDS:                          # this run's networks, unchanged, on LNDb
            EXT.score(cfg, cfg.out, LNDB, device, arms=RUN_ARMS, seed=s)
        if SRES.CLAIM in RUN_ARMS:
            EXTERNAL_REPORT = EXT.report(cfg.out)
        else:
            print(f"LNDb scores of {RUN_ARMS} saved under {Path(cfg.out).name}/external_lndb; "
                  f"the table needs the study's {SRES.CLAIM} scores beside them")
    except FileNotFoundError as e:
        print(f"LNDb is not attached, so the external test is skipped: {e}")

if REPORT is not None:
    # The same numbers as a one-page summary for the write-up (saved, not shown).
    SUMMARY = study_summary.write(REPORT, cohort=spec.name.split("_")[0], out_dir=cfg.out,
                               seed=BASE, seeds_report=SEEDS_REPORT,
                               external_report=EXTERNAL_REPORT,
                               sensitivity_report=SENS_REPORT)

ABLATION_REPORT = None
if set(RUN_ARMS) & set(ABLATIONS):
    print(f"\n{'=' * 28} 5. ablation {'=' * 28}")
    ABLATION_REPORT = SRES.ablation_report(READ, DONE or None)


## What the trained arms learned

These figures are drawn from the saved test predictions, so they need no GPU.

**Hardest cases.** The test cases the panel disputed most. For each report field, the black bar is where
the case really sits among the fold's test cases (its percentile of panel disagreement); the coloured
bars are where each method puts it. A method that tracks the panel puts tall bars where the black bar is tall.

In [ ]:
RES = SRES.load_run(READ, BASE)
if RES:
    fold = DEMO["fold"]
    case_arms = {DISPLAY_NAMES[a]: a for a in DEMO["case_arms"]}
    demo.case_gallery(RES[fold], cfg.processed_dir, spec.field_names, case_arms, ships,
                         n_cases=DEMO["n_cases"],
                         title=f"{spec.name}, fold {fold}: the most disputed test cases")
    plt.show()

**Catching the disputed cases.** If a reader can look at a fixed share of all cases, chosen by each
method's signal, how many of the panel's contested cases (report-level disagreement above `d0`) do they
see? The dashed line is a random choice. Higher is better.

In [ ]:
if RES:
    curve_arms = {DISPLAY_NAMES[a]: a for a in DEMO["curve_arms"]}
    demo.referral_curve(RES, curve_arms, ships, cfg.contested_d, DEMO["budgets"],
                           title=f"{spec.name}: contested cases caught at each reading budget")
    plt.show()

In [ ]:
# Run last, once every result is saved. With RELEASE_RUNTIME_AT_END on Colab this
# disconnects the runtime so the GPU is handed back; on Kaggle and locally it only says so.
from backend.notebooks.shared.colab import release_runtime

if RELEASE_RUNTIME_AT_END:
    release_runtime()
